# EXP001 — Tie Strength Decay Function

**Question:** Which decay function best models relationship warmth from connection recency data?
**Hypothesis:** Exponential decay with half-life ~365 days will outperform linear and step functions because professional relationship warmth decays continuously, not in discrete steps.
**Success criterion:** Winning variant achieves NDCG@10 ≥ 0.80 with margin ≥ 0.03 over second-best.
**Production impact:** `netweave/src/edges.py` → `tie_strength()` function.
**Author:** Chuck Wiley  **Date:** 2026-06-23

In [ ]:
import sys
sys.path.insert(0, ".")

import math
import mlflow
import mlflow.tracking
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lib.niw_graph import load_graph
from lib.niw_metrics import ndcg_at_k, precision_at_k, build_ground_truth
from lib.niw_mlflow import start_run, log_result, compare_runs

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENT_ID = "EXP001"
EXPERIMENT_NAME = "Tie Strength Decay Function"
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
# Always load from shared/ — never modify source data
G = load_graph(
    nodes_path="shared/nodes.parquet",
    edges_path="shared/edges.parquet"
)
print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
def exponential_decay(days: int, half_life: int) -> float:
    return math.exp(-0.693 * days / half_life)

variants = {
    "exp_180":  lambda d: exponential_decay(d, 180),
    "exp_365":  lambda d: exponential_decay(d, 365),   # production baseline
    "exp_730":  lambda d: exponential_decay(d, 730),
    "linear":   lambda d: max(0, 1 - (d / 1095)),      # 3-year linear decay
    "step":     lambda d: 1.0 if d <= 180 else 0.6 if d <= 540 else 0.3,
}

In [ ]:
goal_tags = ["medical_diagnostics", "biotech", "investor"]
relevant_set = build_ground_truth(G, goal_tags)
print(f"Ground truth set size: {len(relevant_set)}")

results = []
for variant_name, decay_fn in variants.items():
    with mlflow.start_run(run_name=f"{EXPERIMENT_ID}_{variant_name}"):
        mlflow.log_params({"decay_function": variant_name})

        # Score each node by tie strength using this decay function
        scores = {}
        for n in G.nodes():
            recency_days = G.nodes[n].get("recency_days", 999)
            scores[n] = decay_fn(recency_days)

        ranked_contacts = sorted(scores, key=scores.get, reverse=True)
        score = ndcg_at_k(ranked_contacts, relevant_set, k=10)
        mlflow.log_metric("ndcg_at_10", score)
        results.append({"variant": variant_name, "score": score})
        print(f"{variant_name}: {score:.4f}")

results_df = pd.DataFrame(results).sort_values("score", ascending=False)
print(results_df)

In [ ]:
import panel as pn
pn.extension()

# Static comparison chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(results_df["variant"], results_df["score"])
ax.set_xlabel("NDCG@10")
ax.set_title(f"{EXPERIMENT_ID}: Tie Strength Decay Function Comparison")
plt.tight_layout()
plt.savefig(f"experiments/EXP001_tie_strength/results.png", dpi=150)
plt.show()

# Interactive Panel slider — intuition beyond the metric numbers
half_life_slider = pn.widgets.IntSlider(name="Half-life (days)", start=90, end=1095, step=30, value=365)

@pn.depends(half_life_slider)
def ranked_output(half_life):
    scores = {nid: exponential_decay(G.nodes[nid].get("recency_days", 999), half_life)
              for nid in G.nodes}
    top = sorted(scores, key=scores.get, reverse=True)[:10]
    return pn.pane.DataFrame(pd.DataFrame([(n, f"{scores[n]:.3f}") for n in top],
                             columns=["node_id", "tie_strength"]))

pn.Column(half_life_slider, ranked_output)

## Result and Decision

**Winner:** none — all five variants tied at NDCG@10 = 0.1312, P@10 = 0.10
**Margin over baseline (exp_365):** 0.000
**Decision:** INCONCLUSIVE

**Root cause:** 4 of the 5 ground-truth nodes (biotech/meddiag/investor domain) are old connections with recency_days in the 3,753–5,726 range, ranking 895–1443 under every decay function. Only one relevant node (andrewtdennis, 42 days) reaches the top 10, landing at rank 5 identically across all variants. When recency-based sorting cannot surface the relevant domain experts, the choice of decay shape is moot.

**Structural finding:** On a domain-specific intro query, `tie_strength` (recency) alone is a weak primary ranking signal. The `composite` score — which blends tie_strength with domain_score and role_fit — is what surfaces old-but-relevant contacts. Decay function shape matters only when recency differentiates the relevant set from the non-relevant set.

**Recommendation:** Retain `exp_365` as the production default (it is the least aggressive exponential and penalizes old contacts least). Re-run this experiment using `composite` as the ranking signal instead of `tie_strength` alone, so that EXP001 tests decay shape within the weighted formula rather than in isolation.

**No production change warranted from this run.**